# SSO Signup Optimization — Signup Method Split Check

Josh shared a query reporting the signup-method (auth provider: email / Google / Apple / Facebook) split for one experiment cell over the allocation window `2026-07-09` to `2026-07-23`. This notebook reproduces that query, checks visitor counts per cell from the allocation table, and then independently sanity-checks the reported **totals** — not the auth-method split itself.

In [1]:
from amphibian import get_data_accessor
def query(sql):
    return get_data_accessor(engines=['presto']).fetch_sql(sql=sql)

# Josh's query

In [2]:
sql = f"""--sql
    WITH allocs AS (
    SELECT
        rand_unit_value AS visitor_id,
        event_ts AS allocated_ts
    FROM ab.current_allocations
    WHERE plan_id = 'db5bede5-a26e-4ffa-8c02-ff823b3577b0'
      AND cell_id = 1
),

visitor_clients AS (
    SELECT DISTINCT
        u.client_id
    FROM curated.user_session_conversion_metrics u
    INNER JOIN allocs a
        ON u.visitor_id = a.visitor_id
    WHERE u.date_in_utc >= DATE '2026-07-09'
      AND u.date_in_utc <= DATE '2026-07-23'
      AND u.client_id IS NOT NULL
),

client_signups AS (
    SELECT
        client.client_id,
        client.signup_at,
        LOWER(
            CASE
                WHEN clients.first_auth_via = ''
                    THEN clients.last_auth_via
                ELSE clients.first_auth_via
            END
        ) AS auth_method,
        MAX(
            CASE
                WHEN client_first_conversion.cancellation_adjusted_first_demand_id IS NOT NULL
                    THEN 1
                ELSE 0
            END
        ) AS first_demand_flag
    FROM visitor_clients vc
    INNER JOIN curated.client_journal__view AS client
        ON vc.client_id = client.client_id
    LEFT JOIN client_service_production.clients
        ON client.client_id = clients.client_id
    LEFT JOIN curated.client_first_conversion
        ON client.client_id = client_first_conversion.client_id
        AND client_first_conversion.cancellation_adjusted_first_demand_id IS NOT NULL
        AND DATE_DIFF(
            'day',
            client.signup_at,
            client_first_conversion.cancellation_adjusted_first_demand_ts
        ) <= 7
    WHERE client.household_primary_client_id IS NULL
      AND LOWER(
          CASE
              WHEN clients.first_auth_via = ''
                  THEN clients.last_auth_via
              ELSE clients.first_auth_via
          END
      ) IN ('email', 'google', 'apple', 'facebook')
    GROUP BY
        client.client_id,
        client.signup_at,
        LOWER(
            CASE
                WHEN clients.first_auth_via = ''
                    THEN clients.last_auth_via
                ELSE clients.first_auth_via
            END
        )
)

SELECT
    auth_method,
    COUNT(DISTINCT client_id) AS total_signups,
    1.0000 * COUNT(DISTINCT client_id)
        / SUM(COUNT(DISTINCT client_id)) OVER ()
        AS pct_of_total_signups,
    COUNT(
        DISTINCT CASE
            WHEN first_demand_flag = 1 THEN client_id
        END
    ) AS total_first_demand,
    1.0000 * COUNT(
        DISTINCT CASE
            WHEN first_demand_flag = 1 THEN client_id
        END
    ) / NULLIF(COUNT(DISTINCT client_id), 0)
        AS pct_with_first_demand_7d
FROM client_signups
GROUP BY auth_method
ORDER BY total_signups DESC
"""

df = query(sql)
df

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,auth_method,total_signups,pct_of_total_signups,total_first_demand,pct_with_first_demand_7d
0,email,22375,0.8105,3815,0.1705
1,google,3462,0.1254,477,0.1378
2,apple,1483,0.0537,169,0.1140
3,facebook,285,0.0103,26,0.0912


In [3]:
# Visitor counts by cell_id for the same plan and date range
query("""--sql
SELECT
    cell_id,
    COUNT (DISTINCT rand_unit_value) AS visitor_count
FROM ab.current_allocations
WHERE
    plan_id = 'db5bede5-a26e-4ffa-8c02-ff823b3577b0'
    AND event_ts >= DATE '2026-07-09'
    AND event_ts <= DATE '2026-07-23'
GROUP BY cell_id
""")

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,cell_id,visitor_count
0,2,45270
1,1,45493
2,3,45770


# Independent sanity check

## Total signups & First Fix requests, 2026-07-09 to 2026-07-23

There's no way to confirm the auth-method **split** above with the data sources used elsewhere in this project — none of those sources carry a signup-method field. What can be checked is whether the **totals** are in a plausible range, using the same visit-anchored methodology (signup-page visit from `curated.product_tracking_events`, `signup_ts`/`request_7d_flag` from `curated.user_session_conversion_metrics`) used to build the signup and First Fix baselines elsewhere in this project.

This query is **not** scoped to a specific experiment cell (unlike the query above, which filters to one `cell_id`) — it covers every new visitor reaching the signup page in the window, i.e. roughly the whole 3-arm experiment population at once. So the numbers below aren't expected to match the single-cell totals above 1:1; the check is whether they're the right **order of magnitude** for one cell to be a plausible ~1/3 share of.

**Also worth flagging:** as of this run, the tail end of the window (roughly 2026-07-17 through 2026-07-23) hasn't had a full 7 days elapse since those signup-page visits, so both this query and the query above are likely slightly undercounting signups/first-fixes for those last few days — the same limitation applies to both, so it shouldn't bias one relative to the other.

In [4]:
sql = """--sql
WITH signup_page_visits AS (
    -- One row per visitor, anchored to their first signup-page visit in the window.
    SELECT
        visitor_id,
        MIN(datetime_in_utc) AS signup_page_ts
    FROM curated.product_tracking_events
    WHERE date_in_utc >= DATE '2026-07-09'
      AND date_in_utc <= DATE '2026-07-23'
      AND url LIKE 'https://www.stitchfix.com/signup%'
    GROUP BY visitor_id
),
visitor_conversion AS (
    -- Not upper-bounded by the window end: a visit on 07-23 can still convert up to 7 days later.
    SELECT
        visitor_id,
        MAX(signup_ts) AS signup_ts,
        MAX(COALESCE(request_7d_flag, 0)) AS first_fix_request_7d_flag
    FROM curated.user_session_conversion_metrics
    WHERE region = 'US'
      AND date_in_utc >= DATE '2026-07-09'
    GROUP BY visitor_id
),
classified AS (
    SELECT
        v.signup_page_ts,
        CASE WHEN c.signup_ts IS NULL OR c.signup_ts >= v.signup_page_ts
             THEN 1 ELSE 0 END AS new_visitor,
        CASE WHEN c.signup_ts >= v.signup_page_ts
              AND c.signup_ts < v.signup_page_ts + INTERVAL '7' DAY
             THEN 1 ELSE 0 END AS signup,
        COALESCE(c.first_fix_request_7d_flag, 0) AS first_fix_request
    FROM signup_page_visits v
    LEFT JOIN visitor_conversion c ON c.visitor_id = v.visitor_id
)
SELECT
    SUM(new_visitor) AS new_visitors,
    SUM(CASE WHEN new_visitor = 1 THEN signup ELSE 0 END) AS total_signups,
    SUM(CASE WHEN new_visitor = 1 THEN first_fix_request ELSE 0 END) AS total_first_fix_requests,
    CAST(SUM(CASE WHEN new_visitor = 1 THEN signup ELSE 0 END) AS DOUBLE)
        / NULLIF(SUM(new_visitor), 0) AS signup_conv_rate,
    CAST(SUM(CASE WHEN new_visitor = 1 THEN first_fix_request ELSE 0 END) AS DOUBLE)
        / NULLIF(SUM(new_visitor), 0) AS first_fix_conv_rate
FROM classified
"""

df = query(sql)
df

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,new_visitors,total_signups,total_first_fix_requests,signup_conv_rate,first_fix_conv_rate
0,144907,75268,14266,0.519423,0.098449
